# FRC Fuel Detection — YOLO v11x (Colab, fuel-only, **v2 強化 augmentation**)

只訓練 **fuel** 一個類別。從原始 3 類 zip 自動過濾 labels（只保留 class 2，重編為 class 0）。

**v2 相比 v1 的差異**：
- 訓練資料 1088 → **1342** 張（session 17 最新 auto export）
- 強化 online augmentation（曝光、角度、copy_paste、mixup）
- RUN_NAME 改為 `frc_fuel_yolo11x_v2`，不覆蓋 v1 的 Drive run 目錄

**事前準備**：
1. 把 `FRC-2026-04-18-auto--yolo-2026-04-22T15-58-55-995Z.zip` 上傳到此帳號 Google Drive：`MyDrive/frc-train/FRC-2026-04-18-auto--yolo-2026-04-22T15-58-55-995Z.zip`
2. Runtime → Change runtime type → **T4 GPU**
3. Runtime → Run all

**本版設定**：模型 `yolo11x.pt`，batch=8，imgsz=640，epochs=50；augmentation `hsv_v=0.6 degrees=15 copy_paste=0.3 mixup=0.15`（詳見訓練 cell）

**產物**：存到 `MyDrive/frc-train/runs/frc_fuel_yolo11x_v2-<timestamp>/`
- `weights/best.pt` / `best.onnx` / `last.pt`
- `results.png` / `results.csv` / `metrics.json` / `confusion_matrix*.png`
- `F1_curve.png` / `P_curve.png` / `R_curve.png` / `PR_curve.png`
- `train_batch*.jpg` / `val_batch*_pred.jpg`
- `run_full.zip` — 整包

## 0. 檢查 GPU

In [ ]:
!nvidia-smi

## 1. 安裝 ultralytics

In [ ]:
!pip install -q 'ultralytics>=8.3.0' onnx onnxsim onnxruntime

## 2. 掛載 Google Drive 並設定路徑

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ===== 可調整參數 =====
ZIP_PATH = '/content/drive/MyDrive/frc-train/FRC-2026-04-18-auto--yolo-2026-04-22T15-58-55-995Z.zip'
MODEL = 'yolo11x.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 8
VAL_RATIO = 0.2
SEED = 42
RUN_NAME = 'frc_fuel_yolo11x_v2'
FUEL_ORIGINAL_CLASS_ID = 2   # 原始 zip 裡 fuel 的 class id（0=red_robot, 1=blue_robot, 2=fuel）
# =====================

DATASET_DIR = Path('/content/dataset')
RUNS_DIR = Path('/content/runs')
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/frc-train/runs')

assert Path(ZIP_PATH).exists(), f'zip 不在 {ZIP_PATH} — 請確認 Drive 路徑'
print('OK, zip size:', Path(ZIP_PATH).stat().st_size // (1024*1024), 'MB')

## 3. 解壓 + 80/20 切 train/val

In [ ]:
import zipfile, shutil, random

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

RAW_DIR = DATASET_DIR / '_raw'
RAW_DIR.mkdir()

print('Extracting zip...')
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW_DIR)

images = sorted((RAW_DIR / 'images').glob('*.jpg'))
print(f'found {len(images)} images')

random.seed(SEED)
random.shuffle(images)
n_val = int(len(images) * VAL_RATIO)
val_set = set(img.stem for img in images[:n_val])
print(f'train: {len(images) - n_val}, val: {n_val}')

for split in ['train', 'val']:
    (DATASET_DIR / split / 'images').mkdir(parents=True)
    (DATASET_DIR / split / 'labels').mkdir(parents=True)

for img in images:
    split = 'val' if img.stem in val_set else 'train'
    shutil.copy2(img, DATASET_DIR / split / 'images' / img.name)
    label = RAW_DIR / 'labels' / (img.stem + '.txt')
    if label.exists():
        shutil.copy2(label, DATASET_DIR / split / 'labels' / label.name)

shutil.rmtree(RAW_DIR)
print('Split done.')
!ls {DATASET_DIR}

## 3.5 過濾 labels 只保留 fuel

把每個 label 檔裡非 fuel 的行刪掉，fuel 的 class id 從 2 重編為 0。沒有 fuel 的圖保留為背景（空 label 或無 label 檔）。

In [ ]:
from pathlib import Path

stats = {'label_files': 0, 'kept_boxes': 0, 'dropped_boxes': 0, 'empty_after': 0}
for split in ['train', 'val']:
    label_dir = DATASET_DIR / split / 'labels'
    for lf in label_dir.glob('*.txt'):
        stats['label_files'] += 1
        new_lines = []
        for line in lf.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            cid = int(parts[0])
            if cid == FUEL_ORIGINAL_CLASS_ID:
                # remap class id → 0
                new_lines.append(' '.join(['0'] + parts[1:]))
                stats['kept_boxes'] += 1
            else:
                stats['dropped_boxes'] += 1
        if new_lines:
            lf.write_text('\n'.join(new_lines) + '\n')
        else:
            # 空 label 檔表示這張圖沒 fuel → 當背景，YOLO 需要檔案存在但內容為空
            lf.write_text('')
            stats['empty_after'] += 1

print('過濾完成：')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 4. 寫 data.yaml (fuel-only, nc=1)

In [ ]:
yaml_text = f"""path: {DATASET_DIR}
train: train/images
val: val/images
nc: 1
names:
  0: fuel
"""
data_yaml = DATASET_DIR / 'data.yaml'
data_yaml.write_text(yaml_text)
print(yaml_text)

## 5. 訓練（v2 強化 augmentation）

T4 + yolo11x + 1342 張 + 50 epochs：預估 ~3–4 小時。

**Augmentation 設計（詳 spec §4.4）**：
- `hsv_v=0.6` 曝光度擾動（逆光/過曝/陰影）
- `degrees=15` 鏡頭傾斜（機器人搖晃）
- `translate=0.2` / `scale=0.6` 位置與距離泛化
- `copy_paste=0.3` fuel 物件複製增加密度
- `mixup=0.15` 兩圖混合
- `flipud=0` **保持 0**（fuel 有重力方向）
- 所有參數都是最大強度、實際訓練每張圖每 epoch 隨機取 `[-max, +max]` 連續值

**過激處置**：若前 5 epoch loss 爆高或 NaN，降 `copy_paste=0.15`、`hsv_v=0.4`、`degrees=10` 重跑。

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR / 'detect'),
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    patience=20,
    verbose=True,
    # ===== v2 強化 online augmentation (spec §4.4) =====
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=15,
    translate=0.2,
    scale=0.6,
    shear=3.0,
    perspective=0.0005,
    flipud=0.0,      # fuel 有重力方向，保持 0
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
)

## 6. 驗證 + 列印 metrics

In [ ]:
best_pt = RUNS_DIR / 'detect' / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(str(best_pt))
m = best_model.val(data=str(data_yaml), imgsz=IMGSZ)
print('\n=== Final metrics (fuel) ===')
print(f'mAP50    : {m.box.map50:.3f}')
print(f'mAP50-95 : {m.box.map:.3f}')
print(f'Precision: {m.box.mp:.3f}')
print(f'Recall   : {m.box.mr:.3f}')

## 7. 匯出 ONNX

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, simplify=True, opset=12)
print('ONNX:', onnx_path)
import os
print('size:', os.path.getsize(onnx_path) // 1024, 'KB')

## 8. 存回 Google Drive（整包 + metrics.json + zip）

In [ ]:
import time, json, shutil
from pathlib import Path

ts = time.strftime('%Y%m%d-%H%M%S')
out_dir = DRIVE_SAVE_DIR / f'{RUN_NAME}-{ts}'
out_dir.mkdir(parents=True, exist_ok=True)

run_dir = RUNS_DIR / 'detect' / RUN_NAME

print(f'Copying entire run dir → {out_dir}')
copied = 0
for src in run_dir.rglob('*'):
    if src.is_file():
        rel = src.relative_to(run_dir)
        dst = out_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1
print(f'  {copied} files copied')

metrics_payload = {
    'run_name': RUN_NAME,
    'timestamp': ts,
    'model': MODEL,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'val_ratio': VAL_RATIO,
    'seed': SEED,
    'dataset': {
        'total_images': len(images),
        'train': len(images) - n_val,
        'val': n_val,
        'classes': ['fuel'],
    },
    'overall': {
        'mAP50': float(m.box.map50),
        'mAP50_95': float(m.box.map),
        'precision': float(m.box.mp),
        'recall': float(m.box.mr),
    },
}

try:
    ap50 = m.box.ap50.tolist() if hasattr(m.box, 'ap50') else []
    ap = m.box.ap.tolist() if hasattr(m.box, 'ap') else []
    p_arr = m.box.p.tolist() if hasattr(m.box, 'p') else []
    r_arr = m.box.r.tolist() if hasattr(m.box, 'r') else []
    metrics_payload['per_class'] = {
        'fuel': {
            'mAP50':    float(ap50[0]) if ap50 else None,
            'mAP50_95': float(ap[0])   if ap   else None,
            'precision': float(p_arr[0]) if p_arr else None,
            'recall':   float(r_arr[0]) if r_arr else None,
        }
    }
except Exception as e:
    print(f'warn: per-class metrics extraction failed: {e}')

(out_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False))
print('metrics.json saved')

zip_basename = str(out_dir / 'run_full')
shutil.make_archive(zip_basename, 'zip', root_dir=str(run_dir))
zip_path = Path(zip_basename + '.zip')
print(f'zip saved: {zip_path} ({zip_path.stat().st_size // (1024*1024)} MB)')

print('\n=== Drive 內容 ===')
for p in sorted(out_dir.rglob('*')):
    if p.is_file():
        rel = p.relative_to(out_dir)
        size_kb = p.stat().st_size / 1024
        print(f'  {rel}  ({size_kb:,.1f} KB)')

print(f'\n全部存在: {out_dir}')

## 9.（可選）在幾張 val 圖上跑推論看結果

In [ ]:
import glob
from IPython.display import Image, display
val_imgs = glob.glob(str(DATASET_DIR / 'val' / 'images' / '*.jpg'))[:6]
preds = best_model.predict(val_imgs, save=True, project='/content/predict', name='sample', exist_ok=True)
for p in sorted(glob.glob('/content/predict/sample/*.jpg'))[:6]:
    display(Image(p, width=640))